# NTT Accelerator — Minimal Board Demo

Smoke-tests the `ntt_top` HLS IP on PYNQ-Z2.

**Pipeline:** NTT(a) → NTT(b) → pointwise mul → INTT(c) = a·b mod (x²⁵⁶+1, 3329)

Requires:
- `bitstream/ntt_bd.bit` and `bitstream/ntt_bd.hwh` (matched base names)
- CTRL AXI-Lite slave at `0x4000_0000` (set in Vivado address editor)

In [5]:
from pathlib import Path
from pynq import Overlay, MMIO, allocate
import numpy as np
import time

# Locate bitstream relative to this notebook (notebooks/ → repo root → bitstream/)
REPO_ROOT = Path('/home/xilinx/jupyter_notebooks/kyber-ntt-fpga')
BITSTREAM  = str(REPO_ROOT / 'bitstream' / 'ntt_bd.bit')

N          = 256
Q          = 3329
CTRL_BASE  = 0x40000000

# AXI-Lite CTRL register offsets
OFF_CTRL = 0x00
OFF_A1   = 0x10;  OFF_A2 = 0x14
OFF_B1   = 0x1C;  OFF_B2 = 0x20
OFF_C1   = 0x28;  OFF_C2 = 0x2C

AP_START = 0x1
AP_DONE  = 0x2
AP_IDLE  = 0x4

In [6]:
# ── 1. Program the PL ────────────────────────────────────────────────────────
ol   = Overlay(BITSTREAM)
ctrl = MMIO(CTRL_BASE, 0x10000)

val = ctrl.read(OFF_CTRL)
print(f'CTRL[0x00] = 0x{val:08x}')
print(f'ap_idle    = {bool(val & AP_IDLE)}   ← expect True')

CTRL[0x00] = 0x00000004
ap_idle    = True   ← expect True


In [7]:
# ── Helper: run one polynomial multiplication ─────────────────────────────────
def ntt_mul(a_coeffs, b_coeffs, timeout_s=2.0):
    """Run ntt_top on the PL. Returns c_coeffs (length N, values mod Q)."""
    a_buf = allocate(shape=(N,), dtype=np.uint16)
    b_buf = allocate(shape=(N,), dtype=np.uint16)
    c_buf = allocate(shape=(N,), dtype=np.uint16)

    a_buf[:] = a_coeffs
    b_buf[:] = b_coeffs
    c_buf[:] = 0

    # Flush dcache — HP port bypasses ARM cache
    a_buf.flush(); b_buf.flush(); c_buf.flush()

    # Write physical addresses (64-bit split into lo/hi; hi=0 on 32-bit PS)
    ctrl.write(OFF_A1, a_buf.physical_address & 0xFFFFFFFF); ctrl.write(OFF_A2, 0)
    ctrl.write(OFF_B1, b_buf.physical_address & 0xFFFFFFFF); ctrl.write(OFF_B2, 0)
    ctrl.write(OFF_C1, c_buf.physical_address & 0xFFFFFFFF); ctrl.write(OFF_C2, 0)

    ctrl.write(OFF_CTRL, AP_START)

    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if ctrl.read(OFF_CTRL) & AP_DONE:
            break
    else:
        raise TimeoutError(f'ap_done never asserted within {timeout_s}s')

    # Invalidate dcache before reading result
    c_buf.invalidate()
    result = c_buf[:].copy()

    a_buf.freebuffer(); b_buf.freebuffer(); c_buf.freebuffer()
    return result

In [8]:
# ── 2. Identity test: a=1, b=1 → c should be 1 ───────────────────────────────
a = np.zeros(N, dtype=np.uint16); a[0] = 1
b = np.zeros(N, dtype=np.uint16); b[0] = 1

c = ntt_mul(a, b)
print('c[0:8] =', c[:8].tolist())
assert c[0] == 1 and np.all(c[1:] == 0), 'Identity test FAILED'
print('Identity test PASSED')

c[0:8] = [1, 0, 0, 0, 0, 0, 0, 0]
Identity test PASSED


In [13]:
# ── 3. Golden-model check ─────────────────────────────────────────────────────
def poly_mul_golden(a, b, n=N, q=Q):
    """Software negacyclic poly mul mod (x^n+1, q)."""
    c = np.zeros(n, dtype=np.int64)
    for i in range(n):
        for j in range(n):
            idx = i + j
            sign = 1 if idx < n else -1
            c[idx % n] = (c[idx % n] + sign * int(a[i]) * int(b[j])) % q
    return c.astype(np.uint16)

rng = np.random.seed(42)                                                                                                               
a = np.random.randint(0, Q, size=N, dtype=np.uint16)
b = np.random.randint(0, Q, size=N, dtype=np.uint16)  

expected = poly_mul_golden(a, b)
got      = ntt_mul(a, b)

mismatches = np.sum(got % Q != expected % Q)
print(f'Mismatches: {mismatches}/{N}')
if mismatches == 0:
    print('Golden-model check PASSED')
else:
    bad = np.where(got % Q != expected % Q)[0]
    for i in bad[:8]:
        print(f'  c[{i}]: got={got[i]}, expected={expected[i]}')

Mismatches: 0/256
Golden-model check PASSED


In [ ]:
# ── 4. Latency measurement ────────────────────────────────────────────────────
a = np.random.randint(0, Q, size=N, dtype=np.uint16)                                                                             
b = np.random.randint(0, Q, size=N, dtype=np.uint16)

REPS = 10
t0 = time.perf_counter()
for _ in range(REPS):
    ntt_mul(a, b)
elapsed_ms = (time.perf_counter() - t0) / REPS * 1000
print(f'Average round-trip latency: {elapsed_ms:.2f} ms  (includes DMA + AXI-Lite overhead)')

AttributeError: 'NoneType' object has no attribute 'integers'